In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.master("local[6]").appName("Local Spark") .config('spark.ui.port', '4040') .getOrCreate()
sc = spark.sparkContext

sc


<SparkContext master=local[6] appName=Local Spark>

In [3]:
# Using a small sized part of the dataset, for now
dataset_path = "/home/jovyan/work/dataset/output/csv/*.csv"
#laptimes_path

rddTelemetry = sc.textFile(dataset_path)
print(f"Number of partitions: {rddTelemetry.getNumPartitions()}")


Number of partitions: 2250


In [4]:
def parseTelemetryRow(row):
    splitted = row.split(",")

    (year, event, session_type, driver_name, lap_number, distance_driver_ahead, driver_ahead, acc_x, acc_y,
     acc_z, brake, distance, drs, gear, rel_distance, rpm, speed, throttle, time, x, y, z) = [x for x in splitted]

    return ((event, session_type, driver_name), (float(acc_y), int(brake), float(rpm), float(speed), float(throttle), rel_distance))

In [5]:
# parsing each row by creating key-value rows
columnNames = rddTelemetry.take(1)
print(columnNames)
rddTelemetryKV = (rddTelemetry \
    .filter(lambda x: x != columnNames[0])
    .map(lambda x: parseTelemetryRow(x))
    .filter(lambda x: x[1][5] != 'None'))

rddTelemetryKV = rddTelemetryKV.map(lambda x: (x[0], (x[1][0],x[1][1],x[1][2],x[1][3],x[1][4])))


['year,event,sessionType,driverName,lapNumber,DistanceToDriverAhead,DriverAhead,acc_x,acc_y,acc_z,brake,distance,drs,gear,rel_distance,rpm,speed,throttle,time,x,y,z']


In [6]:
#print(f"Number of rows: {rddTelemetryKV.count()}")

In [7]:
# aggregating by key based on 95th percentile for lateral acceleration, average on braking, engine rpms while accelerating

# sequencing function on (acc_y, brake, rpm, speed, throttle) to calc average and give as a result (max_acc_y, sum_brake, sum_rpm, rows_throttle_speed_threshold, total_rows)
seqFunc = (
    lambda x, y:
    (x[0] if x[0] > y[0] else y[0],
     x[1] + y[1],
     (x[2] + y[2]) if (y[3] > 120 and y[4] > 35) else x[2],
     (x[3] + 1) if (y[3] > 120 and y[4] > 35) else x[3],
     x[4] + 1)
)

#combining function between partitions
combFunc = (
    lambda x, y:
    (x[0] if x[0] > y[0] else y[0],
     x[1] + y[1],
     x[2] + y[2],
     x[3] + y[3],
     x[4] + y[4])
)

#mapping the values
mapFunc = (
    lambda x:
    (x[0],
     x[1]/x[4],
     x[2]/x[3] if x[3] > 0 else x[1])
)

#accumulator is (max_acc_y, sum_brake, sum_rpm, rows_throttle_speed_threshold, total_rows)
rddDrivingStyleParams = rddTelemetryKV\
    .aggregateByKey((-30.0, 0, 0, 0, 0), seqFunc, combFunc)\
    .mapValues(mapFunc)

# (rddTelemetryKV\
#     .map(lambda x: (x[0], (x[1][1], 1)))\
#     .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))\
#     .mapValues(lambda x: x[0]/x[1])\
#     .collect())

In [10]:
# assigning a "style" label to each row based on some parameters that define the style

# we assume that above average is considered agressive in at least 2 of three categories such as max lateral acceleration and high rpms
# one category higher than the average is considered balanced
# the rest is conservative
# as averages we take 30 m/s^2 as the high threshold, braking percentage 20% and rpms around 10500
acc_y_mean_threshold = rddDrivingStyleParams.map(lambda x: x[1][0]).mean()
brake_mean_threshold = rddDrivingStyleParams.map(lambda x: x[1][1]).mean()
rpms_mean_threshold = rddDrivingStyleParams.map(lambda x: x[1][2]).mean()

def assigningFunc(x):
    score = 0
    if x[0] > acc_y_mean_threshold: score+=1
    if x[1] > brake_mean_threshold: score+=1
    if x[2] > rpms_mean_threshold: score+=1
    if score >= 2: return (x, 'AGRESSIVE')
    elif (score == 1): return (x, 'BALANCED')
    else: return (x, 'CONSERVATIVE')


rddTelemetryWStyle = rddDrivingStyleParams.mapValues(assigningFunc).map(lambda x: (x[0], x[1][1]))#.sortBy(lambda x: x[1], ascending=True).collect()

print(rddTelemetryWStyle)

PythonRDD[14] at RDD at PythonRDD.scala:53


In [12]:
print(rddTelemetryWStyle.collect())

[(('Singapore Grand Prix', 'Qualifying', 'GAS'), 'BALANCED'), (('British Grand Prix', 'Practice 2', 'HUL'), 'CONSERVATIVE'), (('Dutch Grand Prix', 'Qualifying', 'LAW'), 'AGRESSIVE'), (('British Grand Prix', 'Practice 2', 'ALO'), 'AGRESSIVE'), (('Hungarian Grand Prix', 'Practice 3', 'HAM'), 'BALANCED'), (('Monaco Grand Prix', 'Practice 3', 'RUS'), 'AGRESSIVE'), (('Azerbaijan Grand Prix', 'Qualifying', 'ANT'), 'AGRESSIVE'), (('Austrian Grand Prix', 'Race', 'RUS'), 'BALANCED'), (('Hungarian Grand Prix', 'Practice 1', 'TSU'), 'CONSERVATIVE'), (('São Paulo Grand Prix', 'Race', 'BEA'), 'BALANCED'), (('Chinese Grand Prix', 'Qualifying', 'HUL'), 'AGRESSIVE'), (('Australian Grand Prix', 'Practice 3', 'HUL'), 'AGRESSIVE'), (('Japanese Grand Prix', 'Practice 1', 'TSU'), 'BALANCED'), (('Canadian Grand Prix', 'Practice 1', 'LAW'), 'AGRESSIVE'), (('Abu Dhabi Grand Prix', 'Practice 1', 'BEA'), 'AGRESSIVE'), (('São Paulo Grand Prix', 'Qualifying', 'PIA'), 'AGRESSIVE'), (('Australian Grand Prix', 'Prac

In [135]:
#loading the second dataset
laptimes_path = "/home/jovyan/work/dataset/output/laptimes/*.csv"
rddLapTimes = sc.textFile(laptimes_path)
print(f"Number of partitions: {rddLapTimes.getNumPartitions()}")

Number of partitions: 77


In [137]:
def parseLaptimesRow(row):
    splitted = row.split(",")
    print(splitted)
    (event,sessionType,totalLaps,team,driverCode,driverNumber,lap,lapTime,isPersonalBest,position,pitOut,tyreCompound,tyreAgeLaps,stintNumber,sector1Time,sector2Time,sector3Time,speedTrapIntermediate1,speedTrapIntermediate2,speedFinishLine,speedStraight,airTemperature,humidity,atmPressure,isRaining,trackTemperature,windDirection,windSpeed) = [x for x in splitted]

    return ((event, sessionType, driverCode), (float(lapTime) if lapTime != 'None' else 0.0, tyreCompound if tyreCompound != 'None' else '', int(tyreAgeLaps) if tyreAgeLaps != '' else 0, int(lap), int(totalLaps), pitOut))

In [138]:
#parsing and cleaning the second dataset
columnNames = rddLapTimes.take(1)
print(columnNames)
rddLapTimesKV = (rddLapTimes \
                  .filter(lambda x: x != columnNames[0])
                  .map(lambda x: parseLaptimesRow(x))\
                  .filter(lambda x: x[1][0] != 0.0 and x[1][1] != '' and x[1][2] != 0))

['event,sessionType,totalLaps,team,driverCode,driverNumber,lap,lapTime,isPersonalBest,position,pitOut,tyreCompound,tyreAgeLaps,stintNumber,sector1Time,sector2Time,sector3Time,speedTrapIntermediate1,speedTrapIntermediate2,speedFinishLine,speedStraight,airTemperature,humidity,atmPressure,isRaining,trackTemperature,windDirection,windSpeed']


In [139]:
#joining the two datasets hopefully not breeaking anything
joinedRDD = rddTelemetryWStyle.join(rddLapTimesKV)

joinedRDD.top(50)


[(('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (141.589, 'MEDIUM', 4, 4, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (138.286, 'MEDIUM', 3, 9, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (137.497, 'MEDIUM', 3, 3, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (130.416, 'SOFT', 3, 12, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (116.258, 'MEDIUM', 6, 6, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (94.14, 'MEDIUM', 2, 2, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (93.363, 'MEDIUM', 5, 5, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (93.163, 'MEDIUM', 2, 8, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRES

In [140]:
def assignTyreWearFunc(x):
    if x[1][2] > 20: return (x, 'WORN')
    if x[1][2] >= 5: return (x, 'USED')
    if x[1][2] > 1 : return (x, 'NEW')
    else: return (x, 'DROP')

In [147]:
joinedRddLabeled = (joinedRDD.mapValues(assignTyreWearFunc) \
                    .filter(lambda x: x[1][1] != 'DROP')\
                    .mapValues(lambda x: (x[0][0], x[0][1][1], x[1], x[0][1][0], x[0][1][3], x[0][1][4], x[0][1][5], 1))\
                    .mapValues(lambda x: (x[0], x[1], x[2], x[3] - ((100 * (1 - x[4]/x[5])) * 0.03), x[4], x[5], x[6], x[7]))\
                    .filter(lambda x: x[0][1] == 'Race' and x[1][6] == 'None' and x[1][4] > 3))

In [158]:
finalRdd = joinedRddLabeled\
    .map(lambda x: ((x[1][0], x[1][1], x[1][2]), (x[1][3], x[1][7])))\
    .reduceByKey(lambda x,y: (x[0] + y[0], x[1] + y[1])) \
    .mapValues(lambda x: x[0]/x[1])


In [159]:
finalRdd.top(50)


[(('CONSERVATIVE', 'SOFT', 'WORN'), 92.13142356156267),
 (('CONSERVATIVE', 'SOFT', 'USED'), 84.81998896918722),
 (('CONSERVATIVE', 'SOFT', 'NEW'), 83.7398336974489),
 (('CONSERVATIVE', 'MEDIUM', 'WORN'), 85.08229211346587),
 (('CONSERVATIVE', 'MEDIUM', 'USED'), 87.81809255068015),
 (('CONSERVATIVE', 'MEDIUM', 'NEW'), 88.84751990726406),
 (('CONSERVATIVE', 'INTERMEDIATE', 'WORN'), 96.58380731966442),
 (('CONSERVATIVE', 'INTERMEDIATE', 'USED'), 100.02724488771875),
 (('CONSERVATIVE', 'INTERMEDIATE', 'NEW'), 123.13774833531792),
 (('CONSERVATIVE', 'HARD', 'WORN'), 85.50705287804273),
 (('CONSERVATIVE', 'HARD', 'USED'), 88.07316554256744),
 (('CONSERVATIVE', 'HARD', 'NEW'), 93.52151416128272),
 (('BALANCED', 'SOFT', 'WORN'), 84.2180852069814),
 (('BALANCED', 'SOFT', 'USED'), 84.2502206839929),
 (('BALANCED', 'SOFT', 'NEW'), 85.17989600807067),
 (('BALANCED', 'MEDIUM', 'WORN'), 81.74082009501367),
 (('BALANCED', 'MEDIUM', 'USED'), 83.05670293340164),
 (('BALANCED', 'MEDIUM', 'NEW'), 85.5012